# Kaggle Digit Recognizer: 初回提出用ベースライン

本ノートブックは、Kaggle「**Digit Recognizer（手書き数字認識）**」コンペティションで最初の提出ファイルを作成するためのシンプルなベースラインコードです。

### このノートブックで学ぶこと
1. 表データ形式（CSV）の手書き数字画像を読み込み、元の28×28ピクセル画像として画面に表示する方法
2. 機械学習で最も基本的かつ強力な前処理「正規化（0〜1へのスケーリング）」のやり方
3. **ランダムフォレスト（Random Forest）** モデルを用いた学習と検証
4. 混同行列（Confusion Matrix）を用いた誤答パターンの確認
5. Kaggle提出形式（`submission.csv`）の正しい出力とチェック方法

> **所要時間目安:** 約15〜30分（Kaggle Notebook上で「Run All」を押すだけで最後まで完走します）

## 1. 必要なライブラリのインポート
データ操作用の `pandas`, `numpy`、可視化用の `matplotlib`, `seaborn`、そして機械学習モデルとして `scikit-learn` を読み込みます。

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("ライブラリの読み込みが完了しました。")

## 2. データの読み込み
Kaggle環境（`/kaggle/input/digit-recognizer/`）とローカル環境（`../` またはカレントディレクトリ）のどちらでも動作するようにパスを自動判定します。

In [ ]:
# データファイルのパスを検索
candidate_paths = [
    "/kaggle/input/digit-recognizer",
    "../input/digit-recognizer",
    "./data",
    "."
]

data_dir = None
for p in candidate_paths:
    if os.path.exists(os.path.join(p, "train.csv")):
        data_dir = p
        break

if data_dir is None:
    print("【注意】train.csv が見つかりませんでした。Kaggle環境では右ペインの '+ Add Data' から 'Digit Recognizer' が追加されているか確認してください。")
    data_dir = "/kaggle/input/digit-recognizer"
else:
    print(f"データディレクトリを発見しました: {data_dir}")

# データの読み込み
train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))

print(f"訓練データ形状: {train_df.shape} (行数: {train_df.shape[0]}, 列数: {train_df.shape[1]})")
print(f"テストデータ形状: {test_df.shape} (行数: {test_df.shape[0]}, 列数: {test_df.shape[1]})")

## 3. データの探索と可視化 (EDA)
データがどのような中身になっているか、グラフと画像で確認します。

### (1) クラス分布の確認
0から9までの数字がバランス良く含まれているかを確認します。

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x='label', data=train_df, palette='viridis')
plt.title("Class Distribution in Training Set (0 to 9)")
plt.xlabel("Digit Label")
plt.ylabel("Count")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 各数字の件数一覧
print(train_df['label'].value_counts().sort_index())

各数字が約4,000枚ずつ均等に含まれていることが分かります（データが偏っていない「バランスデータ」）。

### (2) 手書き数字画像の可視化
784個のピクセル数値を、元の縦28×横28のグリッドに復元（リシェイプ）して画像として描画してみます。

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    # 1枚分のピクセルデータ（784列）を 28x28 に変換
    image_pixels = train_df.iloc[i, 1:].values.reshape(28, 28)
    image_label = train_df.iloc[i, 0]
    
    plt.imshow(image_pixels, cmap='gray')
    plt.title(f"Label: {image_label}")
    plt.axis('off')

plt.tight_layout()
plt.show()

## 4. 前処理（正規化と訓練・検証分割）

1. **特徴量 $X$ と正解ラベル $y$ の分離:** `label` 列を $y$ に、残り784列を $X$ に分けます。
2. **正規化（スケーリング）:** 0〜255の整数を `255.0` で割って、`0.0〜1.0` の小数に変換します。これにより学習が安定します。
3. **訓練・検証分割（Train/Validation Split）:** 訓練データのうち 20% を「手元の模試」として確保し、提出前に精度を確認できるようにします。

In [ ]:
# 1. 特徴量とラベルの分離
y = train_df['label'].values
X = train_df.drop('label', axis=1).values

# 2. 正規化 (0.0 〜 1.0 にスケーリング)
X = X / 255.0
X_test = test_df.values / 255.0

# 3. 訓練データ(80%)と検証データ(20%)に分割
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"学習用データ: {X_train.shape}")
print(f"検証用データ: {X_val.shape}")
print(f"本番テスト用データ: {X_test.shape}")

## 5. ベースラインモデルの学習（ランダムフォレスト）

今回は、初学者にとってパラメータ調整がほぼ不要で、過学習しにくく、高い正解率（約96%）が手堅く出る **ランダムフォレスト（Random Forest）** を採用します。

> **ランダムフォレストとは:** たくさんの「決定木（条件分岐の木）」を少しずつ異なるデータで学習させ、最後にみんなの多数決で正解を決めるアルゴリズム（アンサンブル学習）です。

In [ ]:
# ランダムフォレストモデルの定義 (決定木を100本作成)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

print("モデルの学習を開始します...")
rf_model.fit(X_train, y_train)
print("学習が完了しました！")

## 6. 検証用データでの評価
手元に残しておいた検証データ（8,400枚）を使って、モデルがどれくらい当てられたか（正解率: Accuracy）を測定します。

In [ ]:
# 検証データへの予測
y_val_pred = rf_model.predict(X_val)

# 正解率の計算
val_acc = accuracy_score(y_val, y_val_pred)
print(f"=== 検証用データでの正解率 (Accuracy): {val_acc:.4f} ({val_acc * 100:.2f}%) ===")

約96%〜97%前後の正解率が出ているはずです！
続いて、**混同行列（Confusion Matrix）** を描画して「どの数字をどう間違えやすいか」を確認してみます。

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Confusion Matrix (Validation Data)")
plt.xlabel("Predicted Label (モデルの予想)")
plt.ylabel("True Label (本当の正解)")
plt.show()

対角線（濃い青のマス）が正解した枚数です。対角線以外の数字がわずかに間違えた例です（例えば、「4」と「9」、「3」と「8」などがたまに誤認識されます）。

## 7. 提出用ファイル (`submission.csv`) の作成

いよいよ本番テストデータ（28,000枚）を予測し、Kaggle提出用のCSVファイルを出力します。

> **重要ルール:**
> 1. 列名は `ImageId,Label`
> 2. `ImageId` は 1 から始まる 28,000行
> 3. 欠損値（NaN）がないこと

In [ ]:
# 本番テストデータの予測
test_preds = rf_model.predict(X_test)

# 提出用DataFrameの構築 (ImageIdは1から始まる連番)
submission_df = pd.DataFrame({
    'ImageId': range(1, len(test_preds) + 1),
    'Label': test_preds
})

# CSVファイルとして保存
submission_df.to_csv('submission.csv', index=False)

print("submission.csv を出力しました！")
print("先頭5行の確認:")
print(submission_df.head())
print(f"行数チェック: {len(submission_df)} 行 (28000行ならOK)")
print(f"欠損値チェック: {submission_df.isnull().sum().sum()} 個 (0ならOK)")

## 8. 次のステップ: 手動での提出手順

お疲れ様でした！これで提出ファイルが完成しました。

### Kaggle Notebook上での提出方法:
1. ノートブック右上の **「Save Version」** ボタン（または右サイドバーの **Output** フォルダ）を確認します。
2. 出力された `submission.csv` の右側にある **「Submit to Competition」** ボタンをクリックします。
3. 数十秒でスコアが計算され、リーダーボードにあなたの名前と正解率（約0.96〜0.97）が記録されます！